[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/laleye/IA-Training-participants/blob/main/module1/session1/tp/TP1/TP1_ma_voix_ma_langue.ipynb)

# TP1 — Ma voix, ma langue : contrôle et analyse de mes enregistrements

**Travail inter-session après la session 1** · ≈ 2 h 30 (dont ≈ 1 h d'enregistrement) · Ticket `M1-S1-07`

Ce notebook accompagne l'énoncé `README.md`. Il :
1. vérifie que vos fichiers respectent le **protocole** (format, durée, saturation, nommage, métadonnées) ;
2. vous fait **analyser la F0** de vos paires minimales ;
3. rassemble votre **mini-rapport**.

Sans vos propres enregistrements, le notebook fonctionne sur le pack d'exemple `minimal_pairs` (voix synthétique),
mais le TP doit être rendu **avec vos enregistrements**.

## 0. Préparation

In [ ]:
# ⏳ Installation (≈ 1 min sur Colab, instantané ensuite). Ne pas modifier.
import importlib.util, subprocess, sys
if importlib.util.find_spec("m1utils") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "m1utils[all] @ git+https://github.com/laleye/IA-Training-participants.git"], check=True)
import m1utils
INFO = m1utils.setup_notebook()

In [ ]:
# 📝 PARAMÈTRES — à adapter
LANGUE = "fon"                                   # "fon" ou "wol" (ou autre code validé par le formateur)
DOSSIER_TP1 = "/content/drive/MyDrive/TP1_nom"   # votre dossier Drive (voir le protocole d'enregistrement)

In [ ]:
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from m1utils.audio import audio_info, load_audio, play
from m1utils.data import load_manifest
from m1utils.features import f0_contour, f0_semitones
from m1utils.preprocess import detect_clipping, estimate_snr
from m1utils.text import normalize_text, out_of_alphabet
from m1utils.viz import compare_f0, plot_f0

if INFO["colab"] and DOSSIER_TP1.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

dossier = Path(DOSSIER_TP1)
MODE_EXEMPLE = not (dossier / "audio").exists()
if MODE_EXEMPLE:
    display(Markdown("⚠️ **Dossier introuvable : mode exemple** (pack `minimal_pairs`). "
                     "Corrigez `DOSSIER_TP1` pour analyser vos enregistrements."))
    exemple = load_manifest("minimal_pairs")
    exemple = exemple[exemple.lang == LANGUE] if (exemple.lang == LANGUE).any() else exemple
    fichiers = [Path(p) for p in exemple.abspath]
    metadata = pd.DataFrame({"id": [p.stem for p in fichiers], "fichier": [p.name for p in fichiers],
                             "texte": exemple.text.tolist(), "langue": exemple.lang.tolist(), "speaker_id": "tts"})
    speakers = pd.DataFrame({"speaker_id": ["tts"], "gender": ["non renseigné"], "age_range": [""], "dialect": [""]})
else:
    fichiers = sorted(p for p in (dossier / "audio").iterdir() if p.suffix.lower() in {".wav", ".flac", ".m4a", ".mp3"})
    metadata = pd.read_csv(dossier / "metadata.csv", dtype=str)
    speakers = pd.read_csv(dossier / "speakers.csv", dtype=str)
print(f"{len(fichiers)} fichiers audio · {len(metadata)} lignes de métadonnées · {len(speakers)} locuteur(s)")

## 1. Contrôles automatiques 🟢
Pour chaque fichier : fréquence d'échantillonnage, canaux, durée, saturation, rapport signal/bruit (SNR) estimé,
respect du nommage `{langue}_{speaker_id}_{id_item}`.

In [ ]:
NOMMAGE = re.compile(rf"^{LANGUE}_spk\d{{2}}_{LANGUE}_\d{{2}}$")
lignes = []
for f in fichiers:
    info = audio_info(f)
    y, sr = load_audio(f, sr=None)
    lignes.append({
        "fichier": f.name, "sr_Hz": info["sr"], "canaux": info["channels"], "durée_s": round(info["duration_s"], 2),
        "saturation_%": round(100 * detect_clipping(y), 2), "SNR_dB": round(estimate_snr(y, sr), 1),
        "nommage_ok": bool(NOMMAGE.match(f.stem)) or MODE_EXEMPLE,
    })
controle = pd.DataFrame(lignes)
def problemes(r):
    regles = [(r.sr_Hz < 16_000, "fs<16k"), (r.canaux > 1, "stéréo"), (not 0.5 <= r["durée_s"] <= 15, "durée"),
              (r["saturation_%"] > 0.1, "saturation"), (r.SNR_dB < 15, "bruit"), (not r.nommage_ok, "nom")]
    return " ".join(label for condition, label in regles if condition)


controle["problèmes"] = controle.apply(problemes, axis=1)
display(controle.style.apply(lambda r: ["background-color: #fde2e1" if r["problèmes"] else "" for _ in r], axis=1))
print(f"Fichiers sans problème : {(controle['problèmes'] == '').sum()} / {len(controle)}")

**❓ Interprétez :** quels fichiers faut-il **réenregistrer** ? Un SNR < 15 dB indique un bruit de fond gênant ;
une saturation > 0,1 % s'entend comme un grésillement. (Un fichier stéréo ou à 44,1 kHz n'est pas grave : on le
convertira en S2.)

✍️ *Votre réponse :* …

## 2. Contrôle des métadonnées 🔵
Vérifiez que : (a) chaque fichier audio a une ligne dans `metadata.csv` ; (b) chaque `speaker_id` de `metadata.csv`
existe dans `speakers.csv` ; (c) les textes ne contiennent pas de caractères hors alphabet.

In [ ]:
ids_audio = {f.stem for f in fichiers}
sans_metadata = ...       # ✍️ À COMPLÉTER : fichiers audio absents de metadata.csv (ensemble d'ids)
locuteurs_inconnus = ...  # ✍️ À COMPLÉTER : speaker_id de metadata absents de speakers.csv
hors_alphabet = ...       # ✍️ À COMPLÉTER : dict {id: caractères hors alphabet} (vides exclus)
print("Fichiers sans métadonnées :", sans_metadata or "aucun ✅")
print("Locuteurs inconnus        :", locuteurs_inconnus or "aucun ✅")
print("Caractères hors alphabet  :", hors_alphabet or "aucun ✅")

## 3. Écoute de contrôle 🟢
Écoutez au moins 5 fichiers **au casque**. Le texte affiché correspond-il exactement à ce qui est prononcé ?

In [ ]:
for r in metadata.head(5).itertuples():
    chemin = next((f for f in fichiers if f.stem == r.id), None)
    if chemin:
        play(chemin, label=f"{r.id} — « {r.texte} »")

## 4. Analyse de la F0 de 3 paires minimales 🔵
Choisissez 3 paires (identifiants des deux mots de chaque paire, colonne `notes` du corpus d'enregistrement).
Pour le **wolof** (pas de tons), comparez plutôt les **durées** des paires longueur/gémination (partie 3 de S1-NB02).

Ajustez `FMIN`/`FMAX` à votre voix : homme ≈ 75–300 Hz, femme ≈ 120–450 Hz.

In [ ]:
FMIN, FMAX = 75, 400
if MODE_EXEMPLE:
    PAIRES = [tuple(g.id) for _, g in exemple.groupby("pair")][:3]
else:
    PAIRES = [("fon_spk01_fon_01", "fon_spk01_fon_02"),   # ✍️ remplacez par VOS identifiants
              ("fon_spk01_fon_05", "fon_spk01_fon_06"),
              ("fon_spk01_fon_07", "fon_spk01_fon_08")]

chemins = {f.stem: f for f in fichiers}
textes = dict(zip(metadata["id"], metadata["texte"]))
mesures = []
for a, b in PAIRES:
    items = []
    for i in (a, b):
        y, sr = load_audio(chemins[i])
        items.append((textes.get(i, i), y))
        t, f0 = f0_contour(y, sr, fmin=FMIN, fmax=FMAX)
        voise = ~np.isnan(f0)
        mesures.append({"id": i, "texte": textes.get(i, i),
                        "F0_médiane_Hz": float(np.nanmedian(f0)) if voise.any() else np.nan,
                        "F0_début_Hz": float(f0[voise][:3].mean()) if voise.sum() >= 3 else np.nan,
                        "F0_fin_Hz": float(f0[voise][-3:].mean()) if voise.sum() >= 3 else np.nan,
                        "durée_voisée_s": float(voise.sum() * 0.01)})
    compare_f0(items, fmin=FMIN, fmax=FMAX)
    plt.show()
mesures = pd.DataFrame(mesures)
reference = np.nanmedian(mesures["F0_médiane_Hz"])
mesures["demi-tons / médiane"] = f0_semitones(mesures["F0_médiane_Hz"].to_numpy(), reference).round(1)
display(mesures.round(1))

**❓ Interprétez (5 à 8 lignes par paire)** : la mesure confirme-t-elle le contraste attendu (ton haut/bas, contour
montant, durée longue/brève) ? Si non, pourquoi (prononciation, détection de F0, coarticulation, débit) ?

✍️ *Votre réponse :* …

## 5. Transcription API de 5 items 🟢
Pour 5 de vos items, proposez une transcription en API (voir le cours S1, §9) :

| id | texte | transcription API |
|---|---|---|
| ✍️ | | / / |
| ✍️ | | / / |
| ✍️ | | / / |
| ✍️ | | / / |
| ✍️ | | / / |

## 6. Mini-rapport (≈ 1 page) 🟢
1. **Conditions d'enregistrement** (matériel, pièce, nombre de prises).
2. **Bilan qualité** (tableau de la partie 1) : combien de fichiers conformes ? qu'avez-vous réenregistré ?
3. **Observations phonétiques** (partie 4) : ce que la F0 / les durées montrent.
4. **Difficultés** rencontrées et **conseils** pour un futur enregistreur.

✍️ *Votre rapport :* …

## 7. Bonus 🔴 — Deux locuteurs, une même paire
Si vous avez enregistré une deuxième personne (autre genre ou tranche d'âge), comparez la **même** paire chez les deux
locuteurs, en Hz puis en demi-tons relatifs à la médiane de chaque locuteur (`compare_f0(..., semitones=True)`).
Le contraste tonal est-il plus comparable après normalisation ?

In [ ]:
...  # ✍️ À COMPLÉTER

## ✅ Avant de rendre
- [ ] Le notebook est **exécuté** du début à la fin avec **vos** enregistrements (pas en mode exemple).
- [ ] Les fichiers problématiques ont été réenregistrés (ou justifiés).
- [ ] `metadata.csv`, `speakers.csv` et les **consentements** sont dans votre dossier.
- [ ] Les parties 4 (interprétation), 5 (API) et 6 (rapport) sont remplies.

Rendu : dossier Drive `TP1_{nom}/` partagé avec le formateur, **veille de la session 2, 18 h**.